# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [13]:
!git clone https://github.com/Sulamithsingh/flyrank-ml-internship-starter.git

fatal: destination path 'flyrank-ml-internship-starter' already exists and is not an empty directory.


In [14]:
!ls flyrank-ml-internship-starter

AGENTS.md    docs	outputs		  SETUP.md		   work
CLAUDE.md    GUIDE.md	README.md	  skills
data	     LICENSE	requirements.txt  submission
DATA_USE.md  notebooks	scripts		  w03_data_contract.ipynb


In [15]:
import pandas as pd

df = pd.read_csv("flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv")

print(df.shape)
print(df.columns.tolist())

(30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [16]:
df[[
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position"
]].describe()

,days_since_last_update,impressions_90d,ctr,avg_position
count,30000.000000,30000.000000,30000.000000,30000.00000
mean,46.098300,5200.366300,0.510733,16.34238
std,42.078709,16838.019547,3.279162,15.21679
min,1.000000,1.000000,0.000000,0.00000
25%,20.000000,81.000000,0.000000,6.20000
50%,20.000000,731.000000,0.070000,10.80000
75%,104.000000,3615.250000,0.290000,22.30000
max,373.000000,517715.000000,100.000000,245.00000


In [17]:
print("Freshness Tier")
print(df["freshness_tier"].value_counts())

print("\nPosition Tier")
print(df["position_tier"].value_counts())

print("\nImpression Tier")
print(df["impression_tier"].value_counts())

Freshness Tier
freshness_tier
0-30      20480
91-180     9171
31-90       175
181+        174
Name: count, dtype: int64

Position Tier
position_tier
page_1      11814
striking     7304
page_3_5     7242
top_3        2321
deep         1319
Name: count, dtype: int64

Impression Tier
impression_tier
low          11248
moderate     10469
good          7205
excellent     1078
Name: count, dtype: int64


In [18]:
# Baseline Rule

rule = """
A page should be reviewed if it:
1. Has not been updated for a long time.
2. Still receives a good number of impressions.
3. Has a low click-through rate.
4. Already ranks within a reasonable search position.
"""

print(rule)

reason_codes = {
    "STALE_CONTENT": "Page has not been updated recently.",
    "HIGH_VISIBILITY": "Page receives many impressions.",
    "LOW_CTR": "Many people see the page but few click it.",
    "POSITION_OPPORTUNITY": "The page already ranks reasonably well and can improve."
}

reason_codes


A page should be reviewed if it:
1. Has not been updated for a long time.
2. Still receives a good number of impressions.
3. Has a low click-through rate.
4. Already ranks within a reasonable search position.



{'STALE_CONTENT': 'Page has not been updated recently.',
 'HIGH_VISIBILITY': 'Page receives many impressions.',
 'LOW_CTR': 'Many people see the page but few click it.',
 'POSITION_OPPORTUNITY': 'The page already ranks reasonably well and can improve.'}

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [19]:
import os
import numpy as np

baseline = df.copy()

baseline["score"] = (
    (baseline["days_since_last_update"] >= 90).astype(int) * 30 +
    (baseline["impressions_90d"] >= 1000).astype(int) * 30 +
    (baseline["ctr"] <= 0.10).astype(int) * 20 +
    ((baseline["avg_position"] > 0) & (baseline["avg_position"] <= 20)).astype(int) * 20
)

def get_reason(row):
    reasons = []

    if row["days_since_last_update"] >= 90:
        reasons.append("STALE_CONTENT")

    if row["impressions_90d"] >= 1000:
        reasons.append("HIGH_VISIBILITY")

    if row["ctr"] <= 0.10:
        reasons.append("LOW_CTR")

    if 0 < row["avg_position"] <= 20:
        reasons.append("POSITION_OPPORTUNITY")

    return ", ".join(reasons)

baseline["reason_code"] = baseline.apply(get_reason, axis=1)

baseline = baseline.sort_values("score", ascending=False)

os.makedirs("work/outputs", exist_ok=True)

baseline.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV saved successfully.")
baseline.head(10)

CSV saved successfully.


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,score,reason_code
4708,content_b115f7c74779,client_19581e27de,30.0,0.02,LOW,0.00,keyword article,transactional,NaN,NaN,...,8.0,2.33,4.08,0.00,excellent,page_1,up,35.9,100,"STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR, POSIT..."
26061,content_56f48894e6cb,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,6982.0,45954.0,...,19.1,2.16,7.50,0.00,moderate,striking,down,-45.4,100,"STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR, POSIT..."
29966,content_77867ed726e1,client_19581e27de,30.0,0.29,LOW,0.08,keyword article,transactional,NaN,NaN,...,12.7,0.00,4.08,0.00,good,striking,down,-28.5,100,"STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR, POSIT..."
29957,content_3b806fcc5b2c,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,6289.0,40609.0,...,4.3,3.70,0.00,0.00,good,page_1,down,-52.3,100,"STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR, POSIT..."
26117,content_442ee9e163f9,client_19581e27de,10.0,0.38,MEDIUM,0.03,keyword article,commercial,NaN,NaN,...,6.5,21.43,20.00,0.00,good,page_1,down,-63.7,100,"STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR, POSIT..."
4748,content_918cbbe942f0,client_19581e27de,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,12.1,0.00,0.00,0.00,moderate,striking,down,-47.9,100,"STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR, POSIT..."
16296,content_c4ab6528d476,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,6599.0,44530.0,...,10.1,0.00,0.00,1.61,moderate,striking,down,-38.7,100,"STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR, POSIT..."
16389,content_a939feaaafb0,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,7881.0,51652.0,...,7.5,0.85,2.33,3.39,good,page_1,down,-55.2,100,"STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR, POSIT..."
46,content_c59d46264834,client_19581e27de,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,6.1,0.00,0.00,0.00,moderate,page_1,down,-47.5,100,"STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR, POSIT..."
16337,content_396217b0dc41,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,2950.0,18786.0,...,10.8,0.00,50.00,0.00,moderate,striking,down,-51.6,100,"STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR, POSIT..."


In [20]:
baseline.columns.tolist()

['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct',
 'score',
 'reason_code']

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [21]:
top20 = baseline.head(20)

for i, (_, row) in enumerate(top20.iterrows(), start=1):
    print(f"{i}.")
    print(f"Action: Review and refresh content")
    print(f"Reason Code: {row['reason_code']}")
    print("Confidence: Medium")
    print("What would make it wrong: Seasonal traffic or temporary ranking changes.")
    print("-" * 70)

1.
Action: Review and refresh content
Reason Code: STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR, POSITION_OPPORTUNITY
Confidence: Medium
What would make it wrong: Seasonal traffic or temporary ranking changes.
----------------------------------------------------------------------
2.
Action: Review and refresh content
Reason Code: STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR, POSITION_OPPORTUNITY
Confidence: Medium
What would make it wrong: Seasonal traffic or temporary ranking changes.
----------------------------------------------------------------------
3.
Action: Review and refresh content
Reason Code: STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR, POSITION_OPPORTUNITY
Confidence: Medium
What would make it wrong: Seasonal traffic or temporary ranking changes.
----------------------------------------------------------------------
4.
Action: Review and refresh content
Reason Code: STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR, POSITION_OPPORTUNITY
Confidence: Medium
What would make it wrong: Seasonal traff

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [22]:
print("Weak Picks + Leakage Check")

print("\nPossible weak picks:")
print("- Some pages may have seasonal traffic, so refreshing them may not improve performance.")
print("- Pages with high impressions but already satisfactory CTR may not need immediate updates.")
print("- Recently updated pages could still appear due to other signals.")

print("\nLeakage Check:")
print("- No product flags were used.")
print("- No future-window metrics were used.")
print("- The rule only uses current content features:")
print("  days_since_last_update")
print("  impressions_90d")
print("  ctr")
print("  avg_position")
print("- trend_direction and trend_pct were intentionally NOT used because they can leak future information.")

Weak Picks + Leakage Check

Possible weak picks:
- Some pages may have seasonal traffic, so refreshing them may not improve performance.
- Pages with high impressions but already satisfactory CTR may not need immediate updates.
- Recently updated pages could still appear due to other signals.

Leakage Check:
- No product flags were used.
- No future-window metrics were used.
- The rule only uses current content features:
  days_since_last_update
  impressions_90d
  ctr
  avg_position
- trend_direction and trend_pct were intentionally NOT used because they can leak future information.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.